# UIT-VSFC — Vietnamese Sentiment / Phân loại cảm xúc tiếng Việt

Tôi lưu bài này trong nhật ký học để ôn lại cách viết và đọc code.

**Nguồn / Source:** Bản Colab `02410008-Truong Minh Hoang.ipynb`. Các bài có tên “Bản sao” là tài liệu thực hành tham khảo; tên file không được xem là bằng chứng tác giả.

**Cách dùng / Run:** chạy các cell theo thứ tự. Thư viện được dùng: `collections`, `os`, `pathlib`.

**Execution:** ARCHIVED_SOURCE — bản mã học tập, chưa ghi nhận chạy lại đầy đủ trong đợt này. Output lịch sử được bỏ; hướng dẫn và lỗi cần xử lý nằm trong bảng kỹ thuật.

Learning archive: coursework/tutorial practice, with original code retained for study. Original tutorial references embedded in the source are retained below.


# Phân lớp cảm xúc tiếng Việt trên bộ dữ liệu **UIT-VSFC**

**Yêu cầu*  
- Sử dụng bộ dữ liệu *UIT-VSFC*: huấn luyện **mô hình truyền thống**.  
- **Chỉ dùng tập dev để tinh chỉnh siêu tham số**, không dùng test cho đến bước đánh giá cuối cùng.  
- **Đánh giá** mô hình trên tập **test**.  
- **Phân tích các dự đoán sai** của mô hình *(bỏ qua các mẫu bị gán nhãn sai – nếu có danh sách mẫu nhiễu nhãn, lọc ra; nếu không có, báo cáo toàn bộ lỗi dự đoán)*.  
- Toàn bộ mô hình/thuật toán được **cài tay**.
Các bước thực hiện
1. Giải nén, đọc 3 tệp `sents.txt`, `sentiments.txt`, `topics.txt` cho mỗi split (*train/dev/test*).
2. Tiền xử lý, tách từ tiếng Việt đơn giản, xây dựng BoW.
3. Cài đặt **Multinomial Naive Bayes** (NB), tinh chỉnh **alpha** trên dev.
4. Cài đặt **Logistic Regression One-vs-Rest** (LR, SGD + L2), tinh chỉnh nhẹ trên dev.
5. Báo cáo **Accuracy / Precision / Recall / F1**, **Confusion Matrix** trên test.
6. **Phân tích lỗi**: Top lỗi có **độ tự tin** cao; **từ khóa chỉ báo** (NB) và **trọng số** (LR).


In [ ]:

import os, zipfile, re, math, random, json
from collections import Counter, defaultdict
from pathlib import Path

# Tuỳ chọn: dùng pandas để hiển thị bảng
try:
    import pandas as pd
except Exception as e:
    pd = None
    print("Không thể import pandas; notebook vẫn chạy nhưng bảng sẽ in dạng text.")

zip_path = "/mnt/data/UIT-VSFC.zip"
data_dir = Path("/mnt/data/UIT-VSFC")
if not data_dir.exists():
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(data_dir)

base = data_dir / "UIT-VSFC"
print("Cấu trúc:", os.listdir(base))
print("Train files:", os.listdir(base / "train"))


In [ ]:

def read_split(split):
    with open(base/ split / "sents.txt", "r", encoding="utf-8") as f:
        sents = [line.strip() for line in f if line.strip()]
    with open(base/ split / "sentiments.txt", "r", encoding="utf-8") as f:
        labels = [line.strip() for line in f if line.strip()]
    with open(base/ split / "topics.txt", "r", encoding="utf-8") as f:
        topics = [line.strip() for line in f if line.strip()]
    assert len(sents) == len(labels) == len(topics)
    return sents, labels, topics

train_sents, train_labels, train_topics = read_split("train")
dev_sents, dev_labels, dev_topics = read_split("dev")
test_sents, test_labels, test_topics = read_split("test")

print("Sizes:", len(train_sents), len(dev_sents), len(test_sents))
print("Label set:", sorted(set(train_labels)|set(dev_labels)|set(test_labels)))
label_names = {'0': 'negative', '1': 'neutral', '2': 'positive'}


In [ ]:

# Tokenization tiếng Việt đơn giản + Bag-of-Words

VN_CHARS = "a-zàáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ"
TOKEN_RE = re.compile(rf"[{VN_CHARS}0-9]+", re.IGNORECASE)

def tokenize(text: str):
    text = text.lower()
    return TOKEN_RE.findall(text)

def build_vocab(texts, min_freq=2, max_size=None):
    freq = Counter()
    for t in texts:
        freq.update(tokenize(t))
    items = [(w,c) for w,c in freq.items() if c >= min_freq]
    items.sort(key=lambda x:(-x[1], x[0]))
    if max_size is not None:
        items = items[:max_size]
    vocab = {w:i for i,(w,_) in enumerate(items)}
    return vocab, freq

vocab, freq = build_vocab(train_sents, min_freq=2, max_size=40000)
print("Vocab size:", len(vocab))

def featurize(text, vocab):
    bow = Counter()
    for tok in tokenize(text):
        if tok in vocab:
            bow[tok]+=1
    return bow

def prepare_dataset(sents, labels, vocab):
    return [(featurize(s, vocab), y) for s,y in zip(sents, labels)]

train_data = prepare_dataset(train_sents, train_labels, vocab)
dev_data   = prepare_dataset(dev_sents,   dev_labels,   vocab)
test_data  = prepare_dataset(test_sents,  test_labels,  vocab)


In [ ]:

def evaluate_preds(y_true, y_pred, labels):
    labels = sorted(labels)
    idx = {c:i for i,c in enumerate(labels)}
    cm = [[0]*len(labels) for _ in labels]
    for t,p in zip(y_true,y_pred):
        cm[idx[t]][idx[p]] += 1
    prec, rec, f1 = {}, {}, {}
    for i,c in enumerate(labels):
        tp = cm[i][i]
        fp = sum(cm[r][i] for r in range(len(labels))) - tp
        fn = sum(cm[i][c2] for c2 in range(len(labels))) - tp
        precision = tp / (tp+fp) if (tp+fp)>0 else 0.0
        recall    = tp / (tp+fn) if (tp+fn)>0 else 0.0
        f1[c] = 2*precision*recall/(precision+recall) if (precision+recall)>0 else 0.0
        prec[c]=precision; rec[c]=recall
    acc = sum(1 for t,p in zip(y_true,y_pred) if t==p)/len(y_true)
    return {"acc":acc,"prec":prec,"rec":rec,"f1":f1,"cm":cm,"labels":labels}

def pretty_report(rep, label_names):
    print(f"Accuracy: {rep['acc']:.4f}")
    print("Per-class metrics:")
    for c in rep["labels"]:
        print(f"  {c:>1} ({label_names.get(c,c):>8})  P={rep['prec'][c]:.4f}  R={rep['rec'][c]:.4f}  F1={rep['f1'][c]:.4f}")
    print("Confusion Matrix (rows=true, cols=pred):")
    header = "      " + " ".join([f"{label_names.get(c,c):>9}" for c in rep["labels"]])
    print(header)
    for i,tr in enumerate(rep["labels"]):
        row = "true_"+label_names.get(tr,tr)
        vals = " ".join([f"{rep['cm'][i][j]:>9d}" for j,_ in enumerate(rep['labels'])])
        print(f"{row:>9} {vals}")


In [ ]:

class MultinomialNB:
    def __init__(self, alpha=1.0, vocab=None):
        self.alpha = alpha
        self.vocab = vocab
        self.class_log_prior_ = {}
        self.feature_log_prob_ = {}

    def fit(self, data):
        class_counts = Counter(y for _,y in data)
        total = sum(class_counts.values())
        self.classes_ = sorted(class_counts.keys())
        for c in self.classes_:
            self.class_log_prior_[c] = math.log(class_counts[c] / total)
        word_counts_per_class = {c: Counter() for c in self.classes_}
        for bow,y in data:
            word_counts_per_class[y].update(bow)
        total_words_per_class = {c: sum(word_counts_per_class[c].values()) for c in self.classes_}
        V = len(self.vocab)
        for c in self.classes_:
            self.feature_log_prob_[c] = {}
            denom = total_words_per_class[c] + self.alpha * V
            for w in self.vocab:
                count = word_counts_per_class[c][w]
                self.feature_log_prob_[c][w] = math.log((count + self.alpha) / denom)
        return self

    def predict_log_proba(self, bow):
        log_probs = {}
        for c in self.classes_:
            logp = self.class_log_prior_[c]
            for w,count in bow.items():
                if w in self.feature_log_prob_[c]:
                    logp += count * self.feature_log_prob_[c][w]
            log_probs[c] = logp
        return log_probs

    def predict(self, bow):
        lp = self.predict_log_proba(bow)
        c = max(lp.items(), key=lambda x:x[1])[0]
        # margin = top1 - top2 for confidence
        tops = sorted(lp.values(), reverse=True)
        margin = tops[0] - (tops[1] if len(tops)>1 else -1e9)
        return c, margin

def evaluate_nb(model, dataset, labels_set):
    y_true=[]; y_pred=[]; conf=[]
    for bow,y in dataset:
        p, m = model.predict(bow)
        y_true.append(y); y_pred.append(p); conf.append(m)
    rep = evaluate_preds(y_true, y_pred, labels_set)
    rep["conf"] = conf
    rep["y_true"] = y_true
    rep["y_pred"] = y_pred
    return rep

# Tinh chỉnh alpha trên dev
alphas = [0.1, 0.3, 0.5, 1.0, 1.5, 2.0, 3.0]
best_alpha=None; best_dev=-1
grid=[]
for a in alphas:
    model = MultinomialNB(alpha=a, vocab=vocab).fit(train_data)
    rep = evaluate_nb(model, dev_data, set(train_labels)|set(dev_labels)|set(test_labels))
    grid.append((a, rep["acc"]))
    if rep["acc"]>best_dev:
        best_dev = rep["acc"]; best_alpha=a

print("NB dev grid:", grid)
print("NB best alpha:", best_alpha, "dev acc:", round(best_dev,4))

# Train NB cuối cùng trên train và đánh giá trên test
final_nb = MultinomialNB(alpha=best_alpha, vocab=vocab).fit(train_data)
nb_test_rep = evaluate_nb(final_nb, test_data, set(train_labels)|set(dev_labels)|set(test_labels))
pretty_report(nb_test_rep, label_names)


In [ ]:

# Lỗi tự tin cao (top 50)
rows = []
for i,(t,p,m) in enumerate(zip(nb_test_rep["y_true"], nb_test_rep["y_pred"], nb_test_rep["conf"])):
    if t!=p:
        rows.append((i,t,p,m))
rows.sort(key=lambda x: -x[3])
top_nb_errors = rows[:50]
print("Top 10 lỗi NB có biên độ tự tin lớn:")
for r in top_nb_errors[:10]:
    idx,t,p,m = r
    print(idx, "true=",label_names.get(t,t), "pred=",label_names.get(p,p), "margin=", round(m,4), "|", test_sents[idx])

# Từ chỉ báo theo NB: log p(w|c) - trung bình_c'!=c log p(w|c')
indicative = []
for c in final_nb.classes_:
    for w in vocab:
        v = final_nb.feature_log_prob_[c][w]
        others = [final_nb.feature_log_prob_[oc][w] for oc in final_nb.classes_ if oc!=c]
        score = v - (sum(others)/len(others))
        indicative.append((c,w,score))
indicative.sort(key=lambda x: (-x[2], x[1]))
print("\nTop 20 từ chỉ báo theo lớp (NB):")
by_class = defaultdict(list)
for c,w,s in indicative:
    if len(by_class[c])<20:
        by_class[c].append((w,s))
for c in final_nb.classes_:
    print("Class", c, "(", label_names.get(c,c), ")")
    for w,s in by_class[c]:
        print("  ", w, "->", round(s,4))


In [ ]:

# Logistic Regression One-vs-Rest (SGD + L2)

class LogisticOVR:
    def __init__(self, classes, lr=0.15, epochs=3, l2=1e-4):
        self.classes = sorted(classes)
        self.lr = lr
        self.epochs = epochs
        self.l2 = l2
        self.weights = {c: defaultdict(float) for c in self.classes}
        self.bias = {c: 0.0 for c in self.classes}

    @staticmethod
    def _sigmoid(x):
        if x>=0:
            z = math.exp(-x)
            return 1/(1+z)
        else:
            z = math.exp(x)
            return z/(1+z)

    def fit(self, dataset):
        data = list(dataset)
        for epoch in range(self.epochs):
            random.shuffle(data)
            for bow, y in data:
                for c in self.classes:
                    t = 1.0 if y==c else 0.0
                    s = self.bias[c] + sum(self.weights[c][w]*val for w,val in bow.items())
                    p = self._sigmoid(s)
                    err = (p - t)
                    self.bias[c] -= self.lr * err
                    for w, val in bow.items():
                        grad = err * val + self.l2 * self.weights[c][w]
                        self.weights[c][w] -= self.lr * grad
        return self

    def predict(self, bow):
        scores = {c: self.bias[c] + sum(self.weights[c][w]*val for w,val in bow.items()) for c in self.classes}
        best_c = max(scores.items(), key=lambda x:x[1])[0]
        sorted_scores = sorted(scores.values(), reverse=True)
        margin = sorted_scores[0] - (sorted_scores[1] if len(sorted_scores)>1 else -1e9)
        return best_c, scores, margin

def evaluate_lr(model, dataset, labels_set):
    y_true=[]; y_pred=[]; conf=[]
    for bow,y in dataset:
        p, scores, margin = model.predict(bow)
        y_true.append(y); y_pred.append(p); conf.append(margin)
    rep = evaluate_preds(y_true, y_pred, labels_set)
    rep["conf"]=conf; rep["y_true"]=y_true; rep["y_pred"]=y_pred
    return rep

# Tinh chỉnh nhẹ trên dev
candidates = [(lr,ep,1e-4) for lr in [0.1,0.15] for ep in [2,3]]
best=None; best_acc=-1
for lr,ep,l2 in candidates:
    model = LogisticOVR(classes=set(train_labels)|set(dev_labels)|set(test_labels), lr=lr, epochs=ep, l2=l2).fit(train_data)
    rep = evaluate_lr(model, dev_data, set(train_labels)|set(dev_labels)|set(test_labels))
    if rep["acc"]>best_acc:
        best_acc=rep["acc"]; best=(lr,ep,l2)
best_lr, best_ep, best_l2 = best
print("LR best on dev:", best_lr, best_ep, best_l2, "dev acc:", round(best_acc,4))

# Train & evaluate trên test
final_lr = LogisticOVR(classes=set(train_labels)|set(dev_labels)|set(test_labels), lr=best_lr, epochs=best_ep, l2=best_l2).fit(train_data)
lr_test_rep = evaluate_lr(final_lr, test_data, set(train_labels)|set(dev_labels)|set(test_labels))
pretty_report(lr_test_rep, label_names)


In [ ]:

# Lỗi tự tin cao (LR)
rows = []
for i,(t,p,m) in enumerate(zip(lr_test_rep["y_true"], lr_test_rep["y_pred"], lr_test_rep["conf"])):
    if t!=p:
        rows.append((i,t,p,m))
rows.sort(key=lambda x: -x[3])
top_lr_errors = rows[:50]
print("Top 10 lỗi LR có biên độ điểm lớn:")
for r in top_lr_errors[:10]:
    idx,t,p,m = r
    print(idx, "true=",label_names.get(t,t), "pred=",label_names.get(p,p), "margin=", round(m,4), "|", test_sents[idx])

# Từ có trọng số dương cao nhất cho mỗi lớp (giải thích mô hình)
topK = 20
for c in sorted(set(train_labels)|set(dev_labels)|set(test_labels)):
    sorted_w = sorted(final_lr.weights[c].items(), key=lambda x: -x[1])[:topK]
    print("\nTop", topK, "từ trọng số cao (class", c, "-", label_names.get(c,c), "):")
    for w,weight in sorted_w:
        print(" ", w, "->", round(weight,4))



## Nhận xét & Phân tích

- **Hiệu năng**: Logistic Regression (OVR) sau tinh chỉnh nhỏ trên dev nhỉnh hơn NB (ví dụ trong lần chạy này: *Acc test ≈ 0.878* vs *0.835*).  
- **Nhầm lẫn thường gặp**:  
  - *Trung tính ↔ Tích cực*: câu ngắn, tính từ chung chung (“được”, “ổn”, “tốt”) hoặc thiếu ngữ cảnh.  
  - *Tiêu cực ↔ Trung tính*: các từ phủ định/khuyết thiếu (“chưa”, “cần”, “khó”, “ít”, “trễ”…) khiến câu nghiêng về tiêu cực; NB dễ bị ảnh hưởng khi tần suất từ không đủ mạnh. LR xử lý tốt hơn nhờ trọng số.  
- **Từ khóa chỉ báo**:  
  - NB: trình bày top từ có **log-likelihood ratio** cao theo lớp.  
  - LR: trình bày top **trọng số** dương theo lớp, phản ánh mức đóng góp vào quyết định.  
- **Lỗi dữ liệu (nhiễu nhãn)**: đề bài yêu cầu bỏ qua “mẫu bị gán nhãn sai”. Bộ UIT-VSFC ở đây **không kèm danh sách** ID mẫu nhiễu nhãn → không thể tự động loại bỏ.

## Ghi chú học tập / Study prompts

- Đầu vào của từng hàm là gì, đầu ra có kiểu và kích thước nào?
- Thay đổi tham số hoặc dữ liệu nhỏ sẽ ảnh hưởng kết quả ra sao?
- Cell nào cần dữ liệu bên ngoài, thư viện hoặc thao tác nhập tay?

Explain the data flow, modify one example, and record the result when revisiting this exercise.